# 01 · Bronze — Raw Ingestion

Land every raw source *as-is* into Delta tables in the `bronze` schema, stamping two
lineage columns: the **source file path** and the **ingestion timestamp**. No cleaning
here — bronze is the faithful, replayable copy. (Brief **§1 Ingestion**, **§3 Extraction**.)

The star is **`halfhourly`**: 21 partitioned block files read **in parallel** by Spark
via a single glob — the core "big data" demonstration.

In [ ]:
%run ./00_config_and_setup

In [ ]:
import re
from pyspark.sql import functions as F


def _clean(col: str) -> str:
    """Delta rejects ' ,;{}()\\n\\t=/' in column names. Replace runs of them with '_'
    and trim any trailing '_' (leading '_' kept, e.g. our lineage cols)."""
    return re.sub(r"[ ,;{}()\n\t=/]+", "_", col).rstrip("_")


def ingest(path_key: str, table_name: str, optional: bool = False):
    path = PATHS[path_key]
    if optional and not path_exists(path):
        print(f"skip   {table_name:<16} (no files at {path})")
        return
    df = (read_csv(path)
          .withColumn("_source_file", F.col("_metadata.file_path"))   # serverless/UC-safe
          .withColumn("_ingested_at", F.current_timestamp()))
    df = df.toDF(*[_clean(c) for c in df.columns])                    # Delta-safe column names
    tgt = table("bronze", table_name)
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(tgt)
    print(f"bronze {table_name:<16} rows={spark.table(tgt).count():>12,}  -> {tgt}")

## Fact table — raw half-hourly readings (the large, block-partitioned source)

In [ ]:
ingest("halfhourly", "halfhourly")

## Dimension & reference tables (small)

In [ ]:
ingest("households",    "households")
ingest("holidays",      "bank_holidays")
ingest("weather_daily", "weather_daily")

## Verify the bronze layer (screenshot for §3 Extraction)

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE}"))

In [ ]:
# Peek at the raw half-hourly fact: note the messy 'energy(kWh/hh)' + 7-digit-fraction tstp
display(spark.table(table("bronze", "halfhourly")).limit(5))